# Multi-Armed Bandit Workshop Reflection

**Student:** Chence  
**Topic:** Exploration-exploitation trade-off with epsilon-greedy bandits

This notebook is my cleaned-up submission for the Multi-Armed Bandit workshop. It combines the two competition rounds, the generated CSV files, visualizations, and reflections on why epsilon and constant alpha matter.


## 1. What problem are we solving?

A multi-armed bandit is a simplified reinforcement learning problem. The agent repeatedly chooses one slot machine arm, receives a reward, and updates its estimate of that arm.

The central tension is the **exploration-exploitation trade-off**:

- **Exploration:** try uncertain arms to learn whether they are better.
- **Exploitation:** choose the arm that currently looks best to earn more reward now.

The epsilon-greedy policy handles this by exploring with probability `epsilon` and exploiting with probability `1 - epsilon`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8-whitegrid")


## 2. Helper functions

These functions reproduce the stationary and non-stationary casino simulations used in the workshop.


In [ ]:
def stationary_means(seed_env=42, n_arms=10):
    rng = np.random.default_rng(seed_env)
    return rng.random(n_arms)


def epsilon_greedy(true_means, steps=2000, epsilon=0.1, seed=123):
    rng = np.random.default_rng(seed)
    n_arms = len(true_means)
    q = np.zeros(n_arms)
    counts = np.zeros(n_arms, dtype=int)
    rewards = np.zeros(steps, dtype=int)
    actions = np.zeros(steps, dtype=int)

    for t in range(steps):
        if rng.random() < epsilon:
            action = rng.integers(n_arms)
        else:
            action = int(np.argmax(q))

        reward = int(rng.random() < true_means[action])
        counts[action] += 1
        q[action] += (reward - q[action]) / counts[action]
        rewards[t] = reward
        actions[t] = action

    return rewards, actions, q, counts


def epsilon_greedy_decaying(true_means, steps=2000, eps_start=0.5, eps_end=0.05, seed=123):
    rng = np.random.default_rng(seed)
    n_arms = len(true_means)
    q = np.zeros(n_arms)
    counts = np.zeros(n_arms, dtype=int)
    rewards = np.zeros(steps, dtype=int)
    actions = np.zeros(steps, dtype=int)

    for t in range(steps):
        epsilon = eps_end + (eps_start - eps_end) * max(0, steps - 1 - t) / max(1, steps - 1)
        if rng.random() < epsilon:
            action = rng.integers(n_arms)
        else:
            action = int(np.argmax(q))

        reward = int(rng.random() < true_means[action])
        counts[action] += 1
        q[action] += (reward - q[action]) / counts[action]
        rewards[t] = reward
        actions[t] = action

    return rewards, actions, q, counts


## 3. Round 1 - Stationary Casino

In the stationary setting, each arm has a fixed hidden reward probability. Because the environment does not change, averaging all past rewards is a reasonable way to estimate each arm.


In [ ]:
TRUE_MEANS = stationary_means(seed_env=42, n_arms=10)
STEPS = 2000
SEED_AGENT = 123

print("Hidden true means used for reproducibility:")
print(np.round(TRUE_MEANS, 3))


In [ ]:
epsilons = [0.0, 0.01, 0.05, 0.1, 0.2, 0.5]
runs = {}
rows = []

for eps in epsilons:
    rewards, actions, q, counts = epsilon_greedy(TRUE_MEANS, steps=STEPS, epsilon=eps, seed=SEED_AGENT)
    runs[f"fixed eps={eps}"] = rewards
    rows.append({
        "strategy": f"fixed eps={eps}",
        "total_reward": int(rewards.sum()),
        "best_estimated_arm": int(np.argmax(q)),
        "unique_arms_tried": int((counts > 0).sum()),
    })

rewards_decay, actions_decay, q_decay, counts_decay = epsilon_greedy_decaying(TRUE_MEANS, steps=STEPS, eps_start=0.5, eps_end=0.05, seed=SEED_AGENT)
runs["decay eps=0.5->0.05"] = rewards_decay
rows.append({
    "strategy": "decay eps=0.5->0.05",
    "total_reward": int(rewards_decay.sum()),
    "best_estimated_arm": int(np.argmax(q_decay)),
    "unique_arms_tried": int((counts_decay > 0).sum()),
})

round1_summary = pd.DataFrame(rows).sort_values("total_reward", ascending=False)
round1_summary


![Round 1 epsilon comparison](mab_figures/round1_epsilon_comparison.png)

![Round 1 action counts](mab_figures/round1_action_counts.png)


In [ ]:
plt.figure(figsize=(9, 5))
for label, rewards in runs.items():
    plt.plot(np.cumsum(rewards), label=label)
plt.title("Round 1: Stationary Casino - epsilon comparison")
plt.xlabel("Step")
plt.ylabel("Cumulative reward")
plt.legend(fontsize=8)
plt.show()


### Round 1 reflection

The best result in my repeated comparison was **fixed eps=0.05**, with a total reward of **1925** over 2000 steps.

Epsilon matters because it controls how much randomness the agent keeps in its behavior. With `epsilon = 0`, the agent only exploits what currently looks best. That can work if the early estimates are lucky, but it can also lock onto a bad arm too early. With very high epsilon, such as `0.5`, the agent explores so often that it gives up many chances to use the best-known arm. A moderate epsilon gives the agent enough information to discover strong arms while still collecting rewards from them.

In the stationary casino, the environment is stable, so the sample-average update is appropriate: old rewards and new rewards are samples from the same underlying probability.


## 4. Round 1 CSV generated by the workshop

The workshop also asked us to generate `submissions_round1.csv`. The file below is the local leaderboard produced by running the competition notebook.


In [ ]:
round1_csv = pd.read_csv("submissions_round1.csv")
round1_csv


## 5. Round 2 - Non-stationary Casino

In the non-stationary setting, the reward probabilities drift over time. This makes the problem harder because yesterday's best arm may not be today's best arm.

To adapt, I use a constant step size alpha:

```python
Q[action] = Q[action] + alpha * (reward - Q[action])
```

This is an exponential moving average. It gives more weight to recent rewards, which helps the agent react when the environment changes.


In [ ]:
def step_drift(means, drift_scale=0.01, rng=None):
    means = means + rng.normal(0, drift_scale, size=len(means))
    return np.clip(means, 0.01, 0.99)


def nonstationary_bandit(
    steps=3000,
    n_arms=10,
    eps=0.1,
    alpha=0.1,
    seed_env=2025,
    seed_agent=999,
    drift_scale=0.01,
    use_sample_average=False,
):
    env_rng = np.random.default_rng(seed_env)
    agent_rng = np.random.default_rng(seed_agent)
    means = env_rng.random(n_arms)
    q = np.zeros(n_arms)
    counts = np.zeros(n_arms, dtype=int)
    rewards = np.zeros(steps, dtype=int)
    actions = np.zeros(steps, dtype=int)
    best_probs = np.zeros(steps)
    chosen_probs = np.zeros(steps)

    for t in range(steps):
        means = step_drift(means, drift_scale=drift_scale, rng=env_rng)
        if agent_rng.random() < eps:
            action = agent_rng.integers(n_arms)
        else:
            action = int(np.argmax(q))

        reward = int(agent_rng.random() < means[action])
        counts[action] += 1
        if use_sample_average:
            q[action] += (reward - q[action]) / counts[action]
        else:
            q[action] += alpha * (reward - q[action])

        rewards[t] = reward
        actions[t] = action
        best_probs[t] = float(np.max(means))
        chosen_probs[t] = float(means[action])

    return rewards, actions, q, counts, best_probs, chosen_probs


In [ ]:
configs = [
    ("constant alpha=0.05, eps=0.1", 0.1, 0.05, False),
    ("constant alpha=0.1, eps=0.1", 0.1, 0.1, False),
    ("constant alpha=0.2, eps=0.1", 0.1, 0.2, False),
    ("constant alpha=0.1, eps=0.05", 0.05, 0.1, False),
    ("constant alpha=0.1, eps=0.2", 0.2, 0.1, False),
    ("sample average, eps=0.1", 0.1, 0.1, True),
]

r2_runs = {}
r2_rows = []

for label, eps, alpha, use_avg in configs:
    rewards, actions, q, counts, best_probs, chosen_probs = nonstationary_bandit(eps=eps, alpha=alpha, use_sample_average=use_avg)
    r2_runs[label] = (rewards, actions, best_probs, chosen_probs)
    r2_rows.append({
        "strategy": label,
        "total_reward": int(rewards.sum()),
        "last_500_reward": int(rewards[-500:].sum()),
        "unique_arms_tried": int((counts > 0).sum()),
    })

round2_summary = pd.DataFrame(r2_rows).sort_values("total_reward", ascending=False)
round2_summary


![Round 2 alpha comparison](mab_figures/round2_alpha_comparison.png)

![Round 2 adaptation](mab_figures/round2_adaptation.png)


In [ ]:
plt.figure(figsize=(9, 5))
for label, (rewards, _, _, _) in r2_runs.items():
    plt.plot(np.cumsum(rewards), label=label)
plt.title("Round 2: Non-stationary Casino - constant alpha comparison")
plt.xlabel("Step")
plt.ylabel("Cumulative reward")
plt.legend(fontsize=8)
plt.show()


### Round 2 reflection

The best result in this comparison was **constant alpha=0.2, eps=0.1**, with a total reward of **2656** over 3000 steps.

Constant alpha helps because the environment is drifting. If I use the normal sample average, the estimate of an arm includes very old rewards from when that arm may have had a different reward probability. That makes the estimate slow to change. A constant alpha keeps updating the estimate by a fixed fraction of the latest prediction error, so recent rewards have more influence.

Epsilon still matters in Round 2. If epsilon decays too low, the agent may stop checking other arms and miss a change in the casino. A small but persistent epsilon keeps the agent sampling alternatives, which is useful when the best arm can change.


## 6. Round 2 CSV generated by the workshop

The workshop also asked us to generate `submissions_round2.csv`. The file below is the local leaderboard produced by running the non-stationary competition.


In [ ]:
round2_csv = pd.read_csv("submissions_round2.csv")
round2_csv


## 7. Additional clarification learned with ChatGPT

After discussing the workshop concepts further, I clarified an important distinction: **stationary/non-stationary describes the environment**, while **static/adaptive describes the policy**.

In a **stationary environment**, the true reward probability of each arm stays the same over time. For example, if a casino machine has a fixed long-term payout behavior, then old observations are still useful because the underlying distribution has not changed. This is why the sample-average update works well in Round 1.

In a **non-stationary environment**, the true reward probabilities can drift over time. The arm that was best earlier may not remain best later. In this case, older observations can become misleading, so the agent should give more weight to recent rewards. This is why constant step-size alpha helps in Round 2.

A **static policy** is different: it means the decision rule or selected machine does not change much after an initial observation period. For example, a player might watch ten machines, decide which one looks best, and then keep playing that one. An **adaptive policy** keeps updating estimates and may switch machines when new evidence suggests another arm is better.

For real casinos, regulated slot machines are usually closer to stationary than non-stationary because their RTP or payout configuration is not normally changed frequently. Different machines can still have different long-term payout structures, but short-term winning or losing streaks usually reflect randomness rather than a true change in probability. True non-stationarity would mean the actual payout probabilities change over time, such as through reconfiguration, promotions, faults, or changing external conditions.

In real-world applications, non-stationarity is more common in systems such as advertising, recommendation, A/B testing, and financial markets. User interests, content popularity, conversion rates, and market regimes can change. In those settings, a bandit strategy should usually keep some exploration and use an adaptive update such as constant alpha or a rolling window.

My decision rule is:

- Use a stationary strategy when I believe each arm's reward probability is stable and old data remains useful.
- Use a non-stationary/adaptive strategy when rewards may change over time and recent data should matter more.
- If I am unsure, keep a small amount of exploration and use a modest constant alpha so the agent can adapt without overreacting to noise.


## 8. Final summary

From these two rounds, I learned that there is no single best amount of exploration for every situation. In a stationary casino, the agent can gradually become more confident because the arms do not change. In a non-stationary casino, the agent needs to stay adaptive.

My main takeaways are:

- Low epsilon can earn high reward if the agent finds the best arm early, but it risks premature commitment.
- High epsilon collects more information, but too much random exploration lowers total reward.
- A moderate or decaying epsilon often works well in stationary settings.
- In non-stationary settings, constant alpha is useful because it tracks recent changes better than a simple historical average.
- Persistent exploration is valuable when the environment can drift.
